# Mini-model języka na nazwach chorób
### Sympozjum „Nowoczesne technologie w pediatrii" · Kamil Jędryczek

Ten notatnik pokazuje **ten sam mechanizm co ChatGPT**, tylko w miniaturze (~80 tys. parametrów): sieć przewiduje **następną literę** w łacińskiej nazwie choroby.

**Uruchomienie:** menu `Środowisko wykonawcze` → `Uruchom wszystko` (Ctrl+F9). Trening na CPU zajmuje ok. **15 sekund**.

Krok po kroku poniżej widać to, co na slajdach sekcji LM: tokeny, słownik, pary X/Y, embedding, rozkład następnej litery, temperatura i generowanie.

*Aplikację dla sali uruchamia prelegent skryptem `notebook/uruchom_demo.py` — nie z tego notatnika.*

In [1]:
# ── USTAWIENIA I ZIARNO (powtarzalność na CPU) ─────────────────
import os
import random

os.environ.setdefault("PYTHONHASHSEED", "42")
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

ZBIOR = "choroby"          # można: "leki", "dinozaury"
LICZBA_EPOK = 12
TEMPO_UCZENIA = 0.003
TEMPERATURA = 0.4
ZIARNO = 42

REPO = "https://raw.githubusercontent.com/NXTRSS/sympozjum-pediatria/main/data"
DANE = {
    "choroby": f"{REPO}/choroby.csv",
    "leki": f"{REPO}/leki.csv",
    "dinozaury": "https://raw.githubusercontent.com/junosuarez/dinosaurs/master/dinosaurs.csv",
}

import numpy as np
import pandas as pd

random.seed(ZIARNO)
np.random.seed(ZIARNO)

import tensorflow as tf
from tensorflow import keras

try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass
tf.random.set_seed(ZIARNO)
keras.utils.set_random_seed(ZIARNO)
tf.config.experimental.enable_op_determinism()

E0000 00:00:1795760527.189645   31471 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1795760527.193999   31471 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1795760527.208817   31471 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795760527.208837   31471 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795760527.208839   31471 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795760527.208840   31471 computation_placer.cc:177] computation placer already registered. Please check linka

## Pomocnicze funkcje (tylko wykresy i czytelne wypisywanie)

Poniżej kilka krótkich funkcji **wyłącznie** do wykresów i ładnego druku na ekranie. Wszystkie kroki ML (słownik, X/Y, model, trening, softmax, temperatura, pętla losowania) są w kolejnych komórkach **wprost** — bez importu z modułów projektu.

In [2]:
# ── POMOCNICZE FUNKCJE ────────────────────────────────────────
import matplotlib.pyplot as plt


def wypisz_slownik(znak2id, znaki, wielkosc_slownika):
    print(f"Tokenów w słowniku: {wielkosc_slownika}")
    print("Litery i znaki specjalne:", "".join(znaki))
    print("\nPrzykładowe mapowanie znak → indeks:")
    for z in ("%", "!", " ", "a", "p") + tuple(znaki[:8]):
        if z in znak2id:
            print(f"  {z!r:>4} → {znak2id[z]}")


def wypisz_pary_xy(X, Y, id2znak, sekwencje, ile_par=3):
    for j in range(len(sekwencje)):
        if " " in sekwencje[j]:
            i = j
            break
    else:
        i = 0
    for poz in range(ile_par):
        x_ids = X[i, : poz + 1]
        y_id = Y[i, poz]
        prefiks = "".join(id2znak[k] for k in x_ids if k)
        nast = id2znak[y_id]
        print(f"  X: {prefiks!r}  →  Y: {nast!r}  (indeks {y_id})")


def wykres_krzywej_bledu(historia):
    plt.figure(figsize=(6, 3.5))
    plt.plot(historia.history["loss"], label="błąd treningu", color="#2563eb")
    plt.xlabel("Epoka")
    plt.ylabel("Loss")
    plt.title("Im niżej, tym lepiej model trafia następną literę")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def wypisz_topk_rozklad(prefiks, probs, id2znak, top_k=12):
    top_idx = np.argsort(probs)[-top_k:][::-1]
    print(f"Kontekst: {prefiks!r}")
    for k in top_idx:
        print(f"  P({id2znak[k]!r:>3} | kontekst) = {probs[k]:.3f}")
    return top_idx


def wykres_topk_rozklad(prefiks, probs, id2znak, top_idx):
    labels = [repr(id2znak[k]) for k in top_idx]
    vals = probs[top_idx]
    plt.figure(figsize=(7, 3.5))
    plt.bar(range(len(vals)), vals, color="#7c3aed")
    plt.xticks(range(len(vals)), labels, rotation=45, ha="right")
    plt.ylabel("Prawdopodobieństwo")
    plt.title(f"Top-{len(top_idx)} liter po prefiksie {prefiks!r}")
    plt.tight_layout()
    plt.show()


def wypisz_kroki_generowania(kroki, id2znak, max_wierszy=6):
    print("\nPierwsze kroki (kontekst → wylosowany znak, P):")
    for k, (kont, wybor, p) in enumerate(kroki[:max_wierszy]):
        print(f"  {k+1}. {kont!r} → {id2znak[wybor]!r}  (P≈{p:.3f})")
    if len(kroki) > max_wierszy:
        print(f"  … łącznie {len(kroki)} kroków do !")


def znacznik_nowosci(nazwa, znane):
    return "⚠️ w zbiorze" if nazwa in znane else "✨ nowa"


## 1. Dane — nazwy chorób (slajd *Dane treningowe*)

Uczymy się na liście łacińskich nazw z pliku CSV w repozytorium. Każda nazwa to jeden „dokument" — jak jedno zdanie w dużym modelu językowym.

In [3]:
# ── WCZYTAJ NAZWY ─────────────────────────────────────────────
from pathlib import Path

zrodlo = DANE[ZBIOR]
plik = os.path.basename(zrodlo)
for kandydat in (
    Path(plik),
    Path("data") / plik,
    Path("..") / "data" / plik,
    Path.cwd() / "data" / plik,
    Path.cwd().parent / "data" / plik,
):
    if kandydat.exists():
        zrodlo = str(kandydat)
        break

df = pd.read_csv(zrodlo, header=None, names=["nazwa"], sep="\t")
nazwy = (
    df["nazwa"].dropna().astype(str).str.strip().str.lower().drop_duplicates().tolist()
)
rng = np.random.default_rng(ZIARNO)
nazwy = [nazwy[i] for i in rng.permutation(len(nazwy))]

print(f"Wczytano {len(nazwy)} nazw z: {zrodlo}")
print("Przykłady:", ", ".join(nazwy[:6]))

Wczytano 1292 nazw z: ../data/choroby.csv
Przykłady: ataxia teleangiectasia, cephalhematoma, dermatitis atopica, claudicatio intermittens, adenocarcinoma, trachoma


## 2. Tokenizacja i słownik (slajdy *Tokeny* / *Vocabulary*)

Każdą nazwę opakowujemy w znaki sterujące, jak na slajdach:

- `%` na początku — **START** („zacznij pisać")
- `!` na końcu — **STOP** („koniec nazwy")

Przykład: `pneumonia` → `%pneumonia!`. Spacja, myślnik i apostrof w nazwach też są **tokenami** (pojedynczymi znakami).

In [4]:
# ── SŁOWNIK ZNAKÓW (char → indeks) ─────────────────────────────
sekwencje = ["%" + n + "!" for n in nazwy]
znaki = sorted(set("".join(sekwencje)))
# Indeks 0 zarezerwowany pod padding w Embedding; znaki od 1 w górę
znak2id = {z: i + 1 for i, z in enumerate(znaki)}
id2znak = {i: z for z, i in znak2id.items()}
WIELKOSC_SLOWNIKA = len(znaki) + 1

wypisz_slownik(znak2id, znaki, WIELKOSC_SLOWNIKA)

Tokenów w słowniku: 32
Litery i znaki specjalne:  !%'-abcdefghijklmnopqrstuvwxyz

Przykładowe mapowanie znak → indeks:
   '%' → 3
   '!' → 2
   ' ' → 1
   'a' → 6
   'p' → 21
   ' ' → 1
   '!' → 2
   '%' → 3
   "'" → 4
   '-' → 5
   'a' → 6
   'b' → 7
   'c' → 8


## 3. Kodowanie na liczby (embedding na wejściu)

Model nie widzi liter — widzi **indeksy** ze słownika. To ten sam pomysł co token IDs w ChatGPT, tylko tu token = jedna litera (lub spacja).

In [5]:
# ── PRZYKŁAD: NAZWA → LISTA INDEKSÓW ───────────────────────────
przyklad = nazwy[0]
tekst = "%" + przyklad + "!"
indeksy = [znak2id[z] for z in tekst]
print(f"Nazwa: {przyklad}")
print(f"Tekst ze znacznikami: {tekst}")
print("Indeksy:", indeksy)
print("Z powrotem:", "".join(id2znak[i] for i in indeksy))

Nazwa: ataxia teleangiectasia
Tekst ze znacznikami: %ataxia teleangiectasia!
Indeksy: [3, 6, 25, 6, 29, 14, 6, 1, 25, 10, 17, 10, 6, 19, 12, 14, 10, 8, 25, 6, 24, 14, 6, 2]
Z powrotem: %ataxia teleangiectasia!


## 4. Pary X i Y — „jaka litera następna?" (slajd *X and Y*)

Dla `%otitis externa!` model uczy się m.in.:

- wejście **X** = `%otitis` → cel **Y** = spacja
- **X** = `%otitis extern` → **Y** = `a`
- **X** = `%otitis externa` → **Y** = `!`

Jedno przejście GRU po całej nazwie uczy **wszystkie** prefiksy naraz.

In [6]:
# ── MACIERZE X, Y ───────────────────────────────────────────────
maks_dl = max(len(s) for s in sekwencje)
X = np.zeros((len(sekwencje), maks_dl - 1), dtype="int32")
Y = np.zeros((len(sekwencje), maks_dl - 1), dtype="int32")
for i, s in enumerate(sekwencje):
    ids = [znak2id[z] for z in s]
    X[i, : len(ids) - 1] = ids[:-1]
    Y[i, : len(ids) - 1] = ids[1:]

print(f"Kształt danych: {X.shape[0]} nazw × {X.shape[1]} pozycji")
wypisz_pary_xy(X, Y, id2znak, sekwencje)

Kształt danych: 1292 nazw × 60 pozycji
  X: '%'  →  Y: 'a'  (indeks 6)
  X: '%a'  →  Y: 't'  (indeks 25)
  X: '%at'  →  Y: 'a'  (indeks 6)


## 5. Model — Embedding → GRU → softmax (slajd *RNN cell*)

Architektura wprost w Kerasie:

1. **Embedding(64)** — każdy indeks znaku zamienia się w wektor 64 liczb (uczy się reprezentacji liter)
2. **GRU(128)** z dropout 0.2 — pamięć kontekstu wzdłuż nazwy
3. **Dense + softmax** — rozkład prawdopodobieństwa **następnego** tokenu (litery)

In [7]:
# ── DEFINICJA MODELU ────────────────────────────────────────────
model = keras.Sequential([
    keras.layers.Embedding(
        WIELKOSC_SLOWNIKA,
        64,
        mask_zero=True,
        embeddings_initializer=keras.initializers.GlorotUniform(seed=ZIARNO),
    ),
    keras.layers.GRU(
        128,
        return_sequences=True,
        dropout=0.2,
        seed=ZIARNO,
        kernel_initializer=keras.initializers.GlorotUniform(seed=ZIARNO),
        recurrent_initializer=keras.initializers.Orthogonal(seed=ZIARNO + 1),
        bias_initializer=keras.initializers.Zeros(),
    ),
    keras.layers.Dense(
        WIELKOSC_SLOWNIKA,
        activation="softmax",
        kernel_initializer=keras.initializers.GlorotUniform(seed=ZIARNO + 2),
        bias_initializer=keras.initializers.Zeros(),
    ),
])
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=TEMPO_UCZENIA),
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 6. Trening

Uczymy na **całym** zbiorze (bez osobnego sprawdzianu — notatnik ma być prosty). `LICZBA_EPOK=12`, `TEMPO_UCZENIA=0.003`, ziarno 42.

In [8]:
# ── TRENING ───────────────────────────────────────────────────
historia = model.fit(
    X,
    Y,
    epochs=LICZBA_EPOK,
    shuffle=True,
    verbose=2,
)

Epoch 1/12


41/41 - 3s - 64ms/step - loss: 2.7580


Epoch 2/12


41/41 - 1s - 25ms/step - loss: 2.2888


Epoch 3/12


41/41 - 1s - 25ms/step - loss: 2.1677


... (14 outputs left out)

Epoch 11/12


41/41 - 1s - 24ms/step - loss: 1.5267


Epoch 12/12


41/41 - 1s - 24ms/step - loss: 1.4675


In [9]:
# ── KRZYWA BŁĘDU ──────────────────────────────────────────────
wykres_krzywej_bledu(historia)

/tmp/ipykernel_31471/82951615.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Inferencja — rozkład następnej litery (slajd *Inference*)

Podajemy **prefiks** (np. `%pneumon`) — model zwraca prawdopodobieństwa dla każdego znaku ze słownika. To histogram jak na slajdzie o10.

In [10]:
# ── ROZKŁAD P(następna litera | prefiks) ─────────────────────
prefiks = "%pneumon"
ids = np.array([[znak2id[z] for z in prefiks]])
probs = model(ids, training=False).numpy()[0, -1].astype("float64")
probs[0] = 0
probs /= probs.sum()

top_idx = wypisz_topk_rozklad(prefiks, probs, id2znak, top_k=12)
wykres_topk_rozklad(prefiks, probs, id2znak, top_idx)

Kontekst: '%pneumon'
  P('e' | kontekst) = 0.324
  P('i' | kontekst) = 0.261
  P('a' | kontekst) = 0.210
  P('u' | kontekst) = 0.056
  P('o' | kontekst) = 0.055
  P('y' | kontekst) = 0.032
  P('t' | kontekst) = 0.018
  P('!' | kontekst) = 0.013
  P(' ' | kontekst) = 0.007
  P('c' | kontekst) = 0.004
  P('d' | kontekst) = 0.004
  P('g' | kontekst) = 0.003


/tmp/ipykernel_31471/82951615.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Generowanie krok po kroku (slajd *Sampling*)

1. Zaczynamy od `%`.
2. Model liczy rozkład — **losujemy** literę (temperatura `TEMPERATURA=0.4` = ostrożniejsze losowanie).
3. Dopisujemy literę i powtarzamy, aż wylosujemy `!`.

Poniżej **krótka pętla w notatniku** (nie ukryta w module) + kilka pierwszych kroków wypisanych na ekranie.

In [11]:
# ── JEDNA NAZWA — PĘTLA LOSOWANIA (temperatura) ────────────────
ZNANE = set(nazwy)
kontekst = "%"
kroki = []
maks_znakow = 40

for krok in range(maks_znakow):
    batch = np.array([[znak2id[z] for z in kontekst]])
    rozklad = model(batch, training=False).numpy()[0, -1].astype("float64")
    rozklad = np.log(rozklad + 1e-9) / TEMPERATURA
    rozklad = np.exp(rozklad)
    rozklad[0] = 0
    rozklad /= rozklad.sum()
    wybor = int(np.random.choice(len(rozklad), p=rozklad))
    kroki.append((kontekst, wybor, rozklad[wybor]))
    if id2znak[wybor] == "!":
        break
    kontekst += id2znak[wybor]

pierwsza_nazwa = kontekst[1:]
print("Wygenerowano:", pierwsza_nazwa)
print("W zbiorze treningowym?", pierwsza_nazwa in ZNANE)
wypisz_kroki_generowania(kroki, id2znak)

Wygenerowano: erythema pantrae poli
W zbiorze treningowym? False

Pierwsze kroki (kontekst → wylosowany znak, P):
  1. '%' → 'e'  (P≈0.031)
  2. '%e' → 'r'  (P≈0.043)
  3. '%er' → 'y'  (P≈0.967)
  4. '%ery' → 't'  (P≈0.995)
  5. '%eryt' → 'h'  (P≈0.999)
  6. '%eryth' → 'e'  (P≈0.874)
  … łącznie 22 kroków do !


## 9. Dziesięć nazw — czy model „wymyśla" coś nowego?

Model **zawsze** wybiera następną literę — nigdy nie mówi „nie wiem". Sprawdzamy tylko, czy wylosowana nazwa **już była** w zbiorze (proste `nazwa in ZNANE`).

In [12]:
# ── 10 NAZW — sprawdzenie nowości (nazwa in ZNANE) ────────────
lista_nazw = []
nowe = 0
for _ in range(10):
    kontekst = "%"
    for __ in range(maks_znakow):
        batch = np.array([[znak2id[z] for z in kontekst]])
        rozklad = model(batch, training=False).numpy()[0, -1].astype("float64")
        rozklad = np.log(rozklad + 1e-9) / TEMPERATURA
        rozklad = np.exp(rozklad)
        rozklad[0] = 0
        rozklad /= rozklad.sum()
        wybor = int(np.random.choice(len(rozklad), p=rozklad))
        if id2znak[wybor] == "!":
            break
        kontekst += id2znak[wybor]
    wynik = kontekst[1:]
    lista_nazw.append(wynik)
    znacznik = znacznik_nowosci(wynik, ZNANE)
    if wynik not in ZNANE:
        nowe += 1
    print(f"  • {wynik:<36} {znacznik}")
print(f"\n{nowe}/10 nazw nie występuje w zbiorze treningowym.")

  • carcinoma cortica                    ✨ nowa


  • analitis atuta                       ✨ nowa


  • steatophia contica                   ✨ nowa


  • syndroma conteri                     ✨ nowa


  • syndroma contonica                   ✨ nowa


  • carcinoma praematocalia teriosa      ✨ nowa


  • morbus hiparticularis                ✨ nowa


  • aphalia prograns angiosa             ✨ nowa


  • carcinoma polita                     ✨ nowa


  • anaemia praventriculorum             ✨ nowa

10/10 nazw nie występuje w zbiorze treningowym.
